In [5]:
# -*- coding: utf-8 -*-
"""
miniQMT 回调注册完整示例
说明：该脚本需要独立运行在 Python 环境中（非QMT内置编辑器）
"""

import time
from xtquant.xttrader import XtQuantTrader, XtQuantTraderCallback
from xtquant.xttype import StockAccount
from xtquant.xtconstant import *


In [6]:

# ============================================================
# 1. 定义回调类（继承 XtQuantTraderCallback）
# ============================================================
class MyCallback(XtQuantTraderCallback):
    """
    自定义回调类，重写父类的5个核心方法
    """
    
    def on_stock_order(self, order):
        """委托回调 - 对应 order_callback"""
        print(f"[委托回调] 股票:{order.stock_code}, 订单号:{order.order_id}, "
              f"状态:{order.order_status}, 价格:{order.price}, 数量:{order.quantity}")
        # 可以根据 order.order_status 做逻辑处理 (4:已报, 5:已成, 6:已撤等)

    def on_stock_trade(self, trade):
        """成交回调 - 对应 deal_callback"""
        print(f"[成交回调] 股票:{trade.stock_code}, 成交价:{trade.traded_price}, "
              f"成交量:{trade.traded_volume}, 成交金额:{trade.traded_amount}")

    def on_order_error(self, order_error):
        """委托错误回调 - 对应 orderError_callback"""
        print(f"[错误回调] 订单号:{order_error.order_id}, "
              f"错误码:{order_error.error_id}, 错误信息:{order_error.error_msg}")

    def on_account_status(self, account):
        """账户状态回调 - 对应 account_callback"""
        print(f"[账户回调] 账号:{account.account_id}, 可用资金:{account.available_balance}")

    def on_position(self, position):
        """持仓回调 - 对应 position_callback"""
        print(f"[持仓回调] 股票:{position.stock_code}, 持仓量:{position.volume}, "
              f"成本价:{position.cost_price}")


# ============================================================
# 2. 主程序：初始化交易引擎并注册回调
# ============================================================
if __name__ == "__main__":
    
    # ---------- 必填参数配置 ----------
    # QMT用户数据目录（请替换成您电脑上的实际路径）
    # 路径通常在：C:\Users\你的用户名\AppData\Local\XtQuant\userdata\ 或 QMT安装目录下
#     USER_DATA_PATH = r"D:\国金QMT交易端模拟\userdata_mini"
    USER_DATA_PATH = r"D:\国金QMT交易端模拟\userdata"
    
    # 会话ID（随便取一个整数，用于区分不同的连接）
    SESSION_ID = 123456
    
    # 资金账号（券商柜台账号）
    ACCOUNT_ID = "10503139"  #"6000000058"
#     ACCOUNT_TYPE = xtconstant.ACCOUNT_TYPE_STOCK  # 股票账户
    ACCOUNT_TYPE = SECURITY_ACCOUNT
    
    # ---------- 初始化交易对象 ----------
    # 创建交易引擎实例
    xt_trader = XtQuantTrader(USER_DATA_PATH, SESSION_ID)
    
    # 连接QMT交易服务（确保QMT主程序已登录并打开）
    connect_result = xt_trader.connect()
    if connect_result != 0:
        print(f"连接失败，错误码: {connect_result}")
        print("请检查：1. QMT是否已登录并打开 2. 路径是否正确")
        exit()
    print("连接交易服务成功！")
    
    # ---------- 订阅资金账号 ----------
    account = StockAccount(ACCOUNT_ID, ACCOUNT_TYPE)
    subscribe_result = xt_trader.subscribe(account)
    if subscribe_result != 0:
        print(f"订阅账号失败，错误码: {subscribe_result}")
        exit()
    print(f"订阅账号 {ACCOUNT_ID} 成功！")
    
    # ---------- ★★★ 注册自定义回调 ★★★ ----------
    # 实例化 MyCallback 并注册到交易引擎
    callback = MyCallback()
    xt_trader.register_callback(callback)
    print("回调类 MyCallback 已成功注册！")
    
    # ---------- 测试下单（触发回调） ----------
    # 示例：以市价买入 100 股 贵州茅台（仅用于演示回调触发）
    # 实际测试时建议先用模拟盘或很小的仓位
    print("\n尝试下单以触发回调...")
    order_id = xt_trader.order_stock(
        account=account,
        stock_code="600519.SH",
#         order_type=xtconstant.ORDER_TYPE_BUY,  # 买入
        order_type = STOCK_BUY,
        order_volume=100,
#         price_type=xtconstant.PRICE_TYPE_LIMIT,  # 限价单
        price_type = FIX_PRICE,
        price=1800.00  # 假设价格，请填当前合理价格
    )
    
    if order_id == -1:
        print("下单失败，请检查参数（例如股价超出涨跌幅限制）")
    else:
        print(f"下单成功，订单号: {order_id}")
    
    # ---------- 保持脚本运行，等待回调推送 ----------
    print("\n正在等待回调推送，10秒后自动退出...")
    time.sleep(10)  # 等待回调触发
    print("脚本结束。")

连接失败，错误码: -1
请检查：1. QMT是否已登录并打开 2. 路径是否正确
连接交易服务成功！


AttributeError: 'int' object has no attribute 'upper'